In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading Semantic Retriever for Data Pruning...")
# 1. Load the retriever (same as training)
semantic_retriever = SentenceTransformer("all-MiniLM-L6-v2")

def retrieve_top_k_series(claim, spec, k=3):
    series_list = spec.get("ser", [])
    if not isinstance(series_list, list) or len(series_list) <= k: return spec
    docs = [claim] + [json.dumps(s) for s in series_list]
    with torch.no_grad():
        embs = semantic_retriever.encode(docs, convert_to_tensor=True)
        sims = torch.nn.functional.cosine_similarity(embs[0:1], embs[1:])
    top_k = sims.topk(k).indices.sort().values.tolist()
    spec["ser"] = [series_list[i] for i in top_k]
    spec["is_truncated"] = True
    return spec

def lightweight_prune_for_llm(raw_spec_dict, claim_text):
    """Exactly mirrors the training stage pruning logic."""
    if not isinstance(raw_spec_dict, dict): return {}
    cleaned = raw_spec_dict.copy()
    if 'ser' not in cleaned or not isinstance(cleaned['ser'], list) or len(cleaned['ser']) == 0: return {}

    def compress_val(v):
        if isinstance(v, float): return int(v) if v.is_integer() else round(v, 2)
        return v

    def sweep_for_loops(obj):
        if isinstance(obj, dict):
            for k, v in list(obj.items()):
                if k == 'data': continue 
                if isinstance(v, list) and all(isinstance(x, (str, int, float, bool)) for x in v):
                    obj[k] = [compress_val(x) for i, x in enumerate(v) if i == 0 or x != v[i-1]][:20]
                elif isinstance(v, (dict, list)): sweep_for_loops(v)

    cleaned.pop('legend', None)
    cleaned.pop('tooltip', None)
    sweep_for_loops(cleaned)
    
    # Apply the retriever!
    cleaned = retrieve_top_k_series(claim_text, cleaned, k=3)
    
    topo = cleaned.get('topo', {})
    topo_type_raw = topo.get('type', '') if isinstance(topo, dict) else ''
    
    if isinstance(topo_type_raw, list):
        topo_type = str(topo_type_raw[0]).lower() if len(topo_type_raw) > 0 else ''
    else:
        topo_type = str(topo_type_raw).lower()
        
    for ser in cleaned.get('ser', []):
        if isinstance(ser, dict):
            ser.pop('ds', None); ser.pop('is_subsampled', None); ser.pop('critical_points_retained', None)
            if 'pie' in topo_type: ser.pop('trend', None); ser.pop('stats', None)
            if 'data' in ser and isinstance(ser['data'], list):
                ser['data'] = [[compress_val(v) for v in pt] if isinstance(pt, list) else compress_val(pt) for pt in ser['data']]

    return cleaned

def format_inference_data(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
        
    formatted_data = []
    for item in raw_data:
        claim = item.get("claim", "")
        spec_dict = item.get("extended_spec", item.get("raw_spec", {}))
        
        # Pass BOTH the spec and the claim to match the training script
        cleaned_spec = lightweight_prune_for_llm(spec_dict, claim)
        cleaned_spec_str = json.dumps(cleaned_spec, separators=(',', ':')) if isinstance(cleaned_spec, dict) else str(cleaned_spec)
        
        system_prompt = "You are an expert chart fact-checker. Analyze the JSON specification to verify the user's claim."
        user_prompt = f"Verify this claim.\nClaim: {claim}\nChart Specification: {cleaned_spec_str}"

        raw_chart_type = str(item.get("chart_type", "")).strip().lower()

        # Fall back to topo.type when the dataset field is absent OR is "unknown"
        if not raw_chart_type or raw_chart_type == "unknown":
            topo_type_raw = (
                raw_spec.get("topo", {}).get("type", "")
                if isinstance(raw_spec, dict) else ""
            )
            topo_sub_raw = (
                raw_spec.get("topo", {}).get("sub", "")
                if isinstance(raw_spec, dict) else ""
            )
            topo_type = str(topo_type_raw).strip().lower()
            topo_sub  = str(topo_sub_raw).strip().lower()

            # Normalise topo.type → dataset naming convention
            _TOPO_MAP = {
                ("bar",  "vertical"):   "barchart_vertical",
                ("bar",  "horizontal"): "barchart_horizontal",
                ("bar",  "stacked"):    "barchart_vertical",
                ("bar",  ""):           "barchart_vertical",
                ("line", ""):           "line_chart",
                ("line", "area"):       "line_chart",
                ("pie",  ""):           "pie_chart",
                ("pie",  "donut"):      "pie_chart",
                ("scatter", ""):        "scatter_plot",
                ("bubble",  ""):        "bubble_chart",
                ("map",     ""):        "map_chart",
            }
            raw_chart_type = _TOPO_MAP.get(
                (topo_type, topo_sub),
                _TOPO_MAP.get((topo_type, ""), topo_type if topo_type else "unknown")
            )
        
        formatted_data.append({
            "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}], 
            "label_val": item.get("label", 0),
            "chart_type": raw_chart_type,           
            "reasoning_type": item.get("reasoning_type", "unknown")    
        })
    return Dataset.from_pandas(pd.DataFrame(formatted_data))

# --- Format the data FIRST before loading Qwen ---
print("\nFormatting test splits (Applying Retriever & Pruning)...")
test1_dataset = format_inference_data(TEST1_PATH)
test2_dataset = format_inference_data(TEST2_PATH)

# --- VRAM Safety: Delete Retriever ---
print("Cleaning up Retriever VRAM...")
del semantic_retriever
torch.cuda.empty_cache()

In [ ]:
import json
import re
import torch
import collections
import pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

# --- Configuration ---
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
ADAPTER_PATH = "/workspace/qwen_verifier_lora/best_verifier"

TEST1_PATH = "test_1_w_spec.json"
TEST2_PATH = "test_2_w_spec.json"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Processor and 4-Bit Base Model...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

# 1. Load Base Model in 4-bit (Matches training memory footprint)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# 2. Merge your trained LoRA weights
print(f"Loading LoRA Adapter from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval() # Set to evaluation mode


# --- 4. Evaluation Loop ---
def evaluate_dataset(dataset):
    true_labels, pred_labels = [], []
    chart_types, reasoning_types = [], []
    
    for idx, item in enumerate(dataset):
        text = processor.apply_chat_template(item["messages"], tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], padding=True, return_tensors="pt").to(DEVICE)
        
        generated_ids = model.generate(
                **inputs, 
                max_new_tokens=150, 
                # --- The Fact-Checking Hyperparameters ---
                do_sample=False,              # 1. Forces Greedy Decoding
                repetition_penalty=1.05,      # 2. Prevents infinite reasoning loops
                pad_token_id=processor.tokenizer.pad_token_id
            )
            
        generated_ids_trimmed = generated_ids[0][len(inputs.input_ids[0]):]
        output_text = processor.decode(generated_ids_trimmed, skip_special_tokens=True).strip()
        
        pred_val = 0
        match = re.search(r"Verdict:\s*(SUPPORTED|REFUTED)", output_text, re.IGNORECASE)
        if match:
            if match.group(1).upper() == "SUPPORTED": pred_val = 1
        else:
            if "supported" in output_text.lower(): pred_val = 1
            
        true_labels.append(item["label_val"])
        pred_labels.append(pred_val)
        chart_types.append(item["chart_type"])
        reasoning_types.append(item["reasoning_type"])
        
    acc = accuracy_score(true_labels, pred_labels) * 100
    f1 = f1_score(true_labels, pred_labels, average="macro") * 100
    return acc, f1, true_labels, pred_labels, chart_types, reasoning_types

# --- 5. Fine-Grained Printer Utility ---
def print_fine_grained(labels, preds, metadata_list, title, col_name):
    print(f"\nFine-grained: performance by {title}")
    print("─" * 50)
    print(f"                    Accuracy  Count\n{col_name}")
    
    stats = collections.defaultdict(lambda: {"correct": 0, "total": 0})
    for y_true, y_pred, meta in zip(labels, preds, metadata_list):
        stats[meta]["total"] += 1
        if y_true == y_pred:
            stats[meta]["correct"] += 1
            
    sorted_stats = sorted(stats.items(), key=lambda x: x[1]["total"], reverse=True)
    for k, v in sorted_stats:
        acc = (v["correct"] / v["total"]) * 100
        print(f"{str(k)[:20]:<20} {acc:>7.1f}% {v['total']:>6}")
    print(f"  Total samples accounted for: {len(labels)} / {len(labels)}")

# --- 6. Execution & Matrix Generation ---


print("\nEvaluating Test 1...")
t1_acc, t1_f1, t1_true, t1_pred, t1_charts, t1_reasons = evaluate_dataset(test1_dataset)
print("Evaluating Test 2...")
t2_acc, t2_f1, t2_true, t2_pred, _, _ = evaluate_dataset(test2_dataset)

avg_acc = (t1_acc + t2_acc) / 2
baseline_acc = 75.0
delta = avg_acc - baseline_acc
delta_str = f"+{delta:.1f}% (above)" if delta >= 0 else f"{delta:.1f}% (below)"

# Print the Final Matrix
SEP = "=" * 115
print(f"\n{SEP}")
print("  CHARTCHECK EVALUATION MATRIX — Qwen Generative Verifier (Multi-Adapter)")
print(SEP)
print("                         Model    Task  Test1_Acc  Test1_F1  Test2_Acc  Test2_F1  Avg_Acc  BLEU ROUGE-L BERTScore")
print("          DePlot-DeBERTa-class        C       75.0      75.0       72.5      72.5     73.8     -       -         -")
print("  DePlot-FlanT5-finetune-multi        M       65.7      65.7       65.9      65.8     65.8  17.3    46.3      91.5")
print("         MatCha-finetune-multi        M       59.4      59.1       61.1      60.9     60.2  17.1    37.3      67.8")
print("             GPT4V (Zero-Shot)        M       73.8      73.5       72.0      71.3     72.9  10.0    32.3      89.1")
print(f"ChartSpec + Qwen-CoT (ours)           C       {t1_acc:<4.1f}      {t1_f1:<4.1f}       {t2_acc:<4.1f}      {t2_f1:<4.1f}     {avg_acc:<4.1f}     -       -         -")
print(SEP)

print(f"\n  Avg accuracy vs NLI-DeBERTa baseline : {delta_str}\n")

print(f"Test 1  →  Acc: {t1_acc:.1f}%   Macro-F1: {t1_f1:.1f}")
print(f"Test 2  →  Acc: {t2_acc:.1f}%   Macro-F1: {t2_f1:.1f}")
print(f"Average →  Acc: {avg_acc:.1f}%\n")

print("Test 1 — Per-class report:")
# label=0 is usually REFUTED, label=1 is SUPPORTED
print(classification_report(t1_true, t1_pred, target_names=["REFUTED", "SUPPORTED"], digits=2))

print_fine_grained(t1_true, t1_pred, t1_charts, "chart type (Test 1)", "chart_type\n")
print_fine_grained(t1_true, t1_pred, t1_reasons, "reasoning type (Test 1)", "reasoning_type\n")
all_results = t1_csv + t2_csv
df_export = pd.DataFrame(all_results)
export_path = f"{ADAPTER_PATH}/chartcheck_qualitative_results.csv"
df_export.to_csv(export_path, index=False, encoding='utf-8')

print(f"\n💾 Saved side-by-side qualitative comparison to: {export_path}")